# Patch SAE segmentation: train-K grid on two Kaggle GPUs

Enable Internet and select **T4 x2**. Push this update to GitHub `refactor` first.
No Hugging Face token, login, secret or artifact upload is required.

**SigLIP 2 and DINOv2** on **VOC and ADE20K**. Train separate CSR, CSRv2 and MPSAE
SAEs at **K=3,12,48**, then evaluate every checkpoint at **K=3,12,48**.
CLIP is excluded. This is **36 fits, 108 sparse evaluations and 4 dense references**.
The 36 matched-K conditions are already included in the 108 sparse conditions.

Compared with the previous 60-point suite: 52 more total evaluation points.
With the complete old workspace retained: reuse 12 fits / 40 evaluations and run
**24 new fits / 72 new evaluations**. Old CLIP results are left intact but excluded.
New empty sessions need the original extraction and fitting again.

Two independent worker processes share the job queue, one per GPU. They preserve
batch sizes and optimization recipes, extract each feature bank once, and reuse
it across training/inference K. Only the coordinator writes tables and ZIPs.
Do not assume a full campaign fits a single Kaggle session.


In [ ]:
ENCODER = "all"              # "all", "siglip2", "dinov2"
DATASET = "all"              # "all", "voc", "ade20k"
TRAIN_K = [3, 12, 48]
INFERENCE_K = [3, 12, 48]
DEVICES = ["cuda:0", "cuda:1"]  # For one GPU, use ["cuda:0"]
GIT_REPO = "https://github.com/thienquang204/MPSAE.git"
GIT_BRANCH = "refactor"


In [ ]:
import os, sys, subprocess, json
from pathlib import Path

# Public downloads only. No login or secret lookup.
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HOME"] = "/tmp/mpsae-hf"
os.environ["HF_DATASETS_CACHE"] = "/tmp/mpsae-datasets"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["MPLBACKEND"] = "Agg"
REPO = Path("/kaggle/working/MPSAE")
WORK = Path("/kaggle/working/patch-scene-v2")
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", GIT_BRANCH, "--depth", "1", GIT_REPO, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
os.chdir(REPO)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)
subprocess.run(["nvidia-smi"], check=True)
assert (REPO / "mpsae/patch_scene/__main__.py").exists(), "Push the new code to GitHub first."
assert "def export_results" in (REPO / "mpsae/patch_scene/report.py").read_text(), "Push the local-results update first."
assert "def inference_ks" in (REPO / "mpsae/patch_scene/config.py").read_text(), "Push the inference-sweep update first."
assert (REPO / "mpsae/patch_scene/scheduler.py").exists(), "Push the multi-GPU training-grid update first."


## Isolated environment
Python 3.11 and CUDA 12.6 PyTorch, independent of Kaggle's notebook packages.
Select T4 x2 for two GPU workers. The project guard does not support P100.
Each GPU has its own memory; this scheduler does not pool VRAM or use DDP.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
PYTHON = "/tmp/mpsae-env/bin/python"
if not Path(PYTHON).exists():
    subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.11", "--seed", "/tmp/mpsae-env"], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "--no-cache-dir",
                "torch==2.9.1", "torchvision==0.24.1",
                "--index-url", "https://download.pytorch.org/whl/cu126"], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "--no-cache-dir",
                "-r", "requirements-patch-scene.txt"], check=True)
subprocess.run([PYTHON, "-m", "pip", "check"], check=True)
subprocess.run([PYTHON, "-m", "mpsae.patch_scene", "doctor", "--device", "cuda:0"], check=True)
visible = int(subprocess.check_output([PYTHON, "-c", "import torch; print(torch.cuda.device_count())"], text=True).strip())
assert len(DEVICES) <= visible, f"Requested {len(DEVICES)} GPUs, but only {visible} are visible."
print("Visible GPUs:", visible)


In [ ]:
BASE = [PYTHON, "-u", "-m", "mpsae.patch_scene"]
def read_plan(*arguments):
    return json.loads(subprocess.check_output(BASE + ["plan", *arguments], cwd=REPO, text=True))

settings = read_plan()["settings"].copy()
settings.update(encoders=["siglip2", "dinov2"], train_k=TRAIN_K, inference_k=INFERENCE_K)
WORK.mkdir(parents=True, exist_ok=True)
CONFIG = str(WORK / "kaggle-train-grid.yaml")
writer = "import json,sys,yaml; from pathlib import Path; Path(sys.argv[1]).write_text(yaml.safe_dump(json.loads(sys.argv[2]), sort_keys=False), encoding='utf-8')"
subprocess.run([PYTHON, "-c", writer, CONFIG, json.dumps(settings)], check=True)
PLAN = read_plan("--config", CONFIG)
print(json.dumps(PLAN, indent=2))
assert set(PLAN["encoders"]) == {"siglip2", "dinov2"}
jobs = (len(PLAN["encoders"]) if ENCODER == "all" else 1) * (len(PLAN["datasets"]) if DATASET == "all" else 1)
print("Selected fits:", jobs * 3 * len(PLAN["train_ks"]))
print("Selected evaluations:", jobs * (1 + 3 * len(PLAN["train_ks"]) * len(PLAN["inference_k"])))
print("Worker devices:", DEVICES)


## Run or resume
One worker owns each GPU. Each encoder/dataset is extracted once; ready fitting
jobs are distributed dynamically. Checkpoints at each training K are independent.
Each checkpoint is reused for its inference sweep; dense is measured once per cell.

The coordinator prints assignments and a heartbeat every 20 seconds. Detailed
worker logs are under `campaigns/<campaign>/logs/`; `scheduler.json` tracks jobs.
The ZIP is refreshed after every completed evaluation, including reused results.
Use the same full workspace to resume; a results-only ZIP cannot restore fits.


In [ ]:
import shutil

command = BASE + ["run", "--config", CONFIG, "--encoder", ENCODER, "--dataset", DATASET,
                  "--workspace", str(WORK), "--devices", *DEVICES, "--cpu-threads", "2"]
try:
    subprocess.run(command, cwd=REPO, check=True)
finally:
    saved = WORK / "patch-scene-results.zip"
    if saved.exists():
        shutil.copy2(saved, "/kaggle/working/patch-scene-results.zip")
        print("Download: /kaggle/working/patch-scene-results.zip")


In [ ]:
from IPython.display import FileLink, display

# This cell may also be run after interrupting the training cell.
# Export only combines completed local results; it does not train or evaluate.
subprocess.run([PYTHON, "-m", "mpsae.patch_scene", "export",
                "--config", CONFIG, "--workspace", str(WORK)], check=True)
archive = Path("/kaggle/working/patch-scene-results.zip")
import shutil
shutil.copy2(WORK / "patch-scene-results.zip", archive)
table = WORK / "campaigns" / PLAN["campaign"] / "table"
receipt = json.loads((table / "results.json").read_text())
print(f"Completed: {receipt['completed']}/{receipt['expected']}")
import pandas as pd
display(pd.read_csv(table / "comparisons.csv").round(2))
print("Matched-K tables: results.tex, table_voc.tex, table_ade20k.tex")
print("Full sweeps: inference_sweep.tex, sweep_voc.tex, sweep_ade20k.tex")
# Use a relative link with the notebook working directory at /kaggle/working.
previous = Path.cwd()
try:
    os.chdir(archive.parent)
    display(FileLink(archive.name))
finally:
    os.chdir(previous)
print("Or use Kaggle's file browser: /kaggle/working/patch-scene-results.zip -> Download")


## Download and paper tables
Download `patch-scene-results.zip` before ending the session. It contains CSV/JSON,
settings, main matched-K tables, full cross-K tables, and a LaTeX preview document.
The main table compares separate fits at train K = inference K; the sweep tables
keep both training K and inference K visible. No rows are averaged across checkpoints.
Bold marks the best sparse method for the same dataset/encoder/train K/inference K.
Single-seed results are descriptive; CSRv2's extra training-mask supervision is stated.

No Hugging Face upload occurs. Saving only notebook code does not preserve data.
The ZIP excludes weights/features: retain the full workspace or save notebook outputs
to reuse fits in a later session. See `docs/patch_scene_kaggle.md` for details.
